In [79]:
import requests
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()  # Läser in variabler från .env
API_KEY = os.getenv("API_KEY")

def get_timetable(stop_id, as_dataframe=False):
    """
    Hämtar tidtabellsinformation för en viss hållplats (stop_id)
    från ResRobot-API:et (departureBoard).

    Parametrar:
    -----------
    stop_id : int
        Hållplatsens ID enligt ResRobot (t.ex. 740015578 för Göteborg Korsvägen).
    as_dataframe : bool
        Om True returneras resultatet som en pandas DataFrame,
        annars returneras resultatet som JSON/dict.

    Returnerar:
    -----------
    dict eller pandas.DataFrame
        JSON som dict om as_dataframe=False,
        annars en pandas.DataFrame med relevanta kolumner.
    """

    # 1. Bygg ihop URL med lämplig endpoint
    url = f"https://api.resrobot.se/v2.1/departureBoard?id={stop_id}&format=json&accessId={API_KEY}"
    
    try:
        # 2. Skicka GET-förfrågan
        response = requests.get(url)
        response.raise_for_status()  # Kastar fel om status != 200

        # 3. Extrahera JSON-svar
        data = response.json()

        # 4. Om as_dataframe=True, build DataFrame
        if as_dataframe:
            # 'Departure' är nyckeln i JSON:et som innehåller avgångar
            departures = data.get("Departure", [])
            df = pd.DataFrame(departures)

            # Exempel: plocka ut några relevanta kolumner och döp om dem
            if not df.empty:
                df_clean = df[["name", "stop", "direction", "date", "time"]].copy()
                return df_clean
            else:
                # Om inga avgångar returneras, returnera en tom DataFrame
                return pd.DataFrame()

        else:
            # Returnera raw JSON-data om man vill göra något annat med den
            return data

    except requests.exceptions.RequestException as err:
        print(f"Nätverks- eller HTTP-fel: {err}")
        return None


In [80]:
stop_id = 740015578
# Hämta rådata (JSON)
raw_data = get_timetable(stop_id=stop_id, as_dataframe=False)
print(raw_data.keys())

# Hämta data som DataFrame
df_departures = get_timetable(stop_id=stop_id, as_dataframe=True)
print(df_departures.head())


dict_keys(['Departure', 'TechnicalMessages', 'serverVersion', 'dialectVersion', 'planRtTs', 'requestId'])
                     name                   stop  \
0    Länstrafik - Buss X4     Göteborg Korsvägen   
1  Länstrafik - Spårväg 4     Göteborg Korsvägen   
2  Länstrafik - Spårväg 8  Göteborg Scandinavium   
3    Länstrafik - Buss 18     Göteborg Korsvägen   
4   Länstrafik - Buss 101     Göteborg Korsvägen   

                       direction        date      time  
0            Kungälv resecentrum  2025-01-09  14:53:00  
1  Angered centrum (Göteborg kn)  2025-01-09  14:53:00  
2  Angered centrum (Göteborg kn)  2025-01-09  14:53:00  
3   Bäckebol Norra (Göteborg kn)  2025-01-09  14:53:00  
4          Göteborg Åkareplatsen  2025-01-09  14:53:00  


In [81]:
def access_id_from_location(location):
    """
    Söker efter alla hållplatser som matchar 'location'-strängen
    och skriver ut deras 'name' och 'extId'.
    """
    url = f"https://api.resrobot.se/v2.1/location.name?input={location}&format=json&accessId={API_KEY}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        
        # stopLocationOrCoordLocation innehåller en lista med StopLocation och/eller CoordLocation
        results = data.get("stopLocationOrCoordLocation", [])
        
        print(f"{'Name':<40} {'extId'}")
        print("-" * 60)
        
        # Iterera över alla resultat
        for item in results:
            # item är i form av {"StopLocation": {...}} eller {"CoordLocation": {...}}
            # Vi använder next(iter(item.values())) för att komma åt själva värdet
            stop_info = next(iter(item.values()))
            
            # Bara StopLocation-objekt har 'extId'
            if "StopLocation" in item:
                name = stop_info.get("name", "N/A")
                ext_id = stop_info.get("extId", "N/A")
                print(f"{name:<40} {ext_id}")
                
    except requests.exceptions.RequestException as err:
        print(f"Nätverks- eller HTTP-fel: {err}")


In [94]:
access_id_from_location("Lund Centralstation")

Name                                     extId
------------------------------------------------------------
Lund Centralstation                      740000120
Falun Centralstation                     740000030
Sundsvall Centralstation                 740000130
Östersund Centralstation                 740000123
Linköping Centralstation                 740000009
Luleå Centralstation                     740000144
Norrköping Centralstation                740000007
Malmö Centralstation                     740000003
Hässleholm Centralstation                740000006
Uppsala Centralstation                   740000005


In [83]:

def get_trips(origin_id=740000001, destination_id=740098001):
    """
    Hämtar reseinformation från ResRobot (ResRobot Route Planner).
    origin_id och destination_id är hållplats-ID (extId) för start respektive slut.
    Exempel:
      - Stockholm Centralstation: 740000001
      - Göteborg Centralstation: 740000002 eller 740098001 (beroende på API)
    """
    url = f"https://api.resrobot.se/v2.1/trip?format=json&originId={origin_id}&destId={destination_id}&passlist=true&showPassingPoints=true&accessId={API_KEY}"

    try:
        response = requests.get(url)
        response.raise_for_status() 
        return response.json()
    except requests.exceptions.RequestException as err:
        print(f"Nätverks- eller HTTP-fel: {err}")
        return None


In [84]:


def get_arrivals(stop_id: int, as_dataframe: bool = False) -> pd.DataFrame or dict:
    """
    Hämtar ankomster (ArrivalBoard) för en viss hållplats (stop_id)
    från ResRobot-API:et.

    Parametrar:
    -----------
    stop_id : int
        Hållplatsens ID (extId), t.ex. 740000002 för Göteborg Centralstation
    as_dataframe : bool
        True = returnera resultatet som en pandas DataFrame
        False = returnera rå JSON (dict)

    Returnerar:
    -----------
    dict eller pandas.DataFrame
        - Om as_dataframe=False: en dict med nyckeln 'Arrival' som innehåller ankomstdatan
        - Om as_dataframe=True: en DataFrame med kolumner som 'name', 'origin'/'direction', 'time', 'date' etc.
    """
    url = f"https://api.resrobot.se/v2.1/arrivalBoard?id={stop_id}&format=json&accessId={API_KEY}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if as_dataframe:
            arrivals = data.get("Arrival", [])
            df = pd.DataFrame(arrivals)
            if not df.empty:
                # Välj vilka kolumner du vill visa (kika i df.columns för att se vad som finns)
                columns_to_pick = ["name", "origin", "time", "date"]
                # Om "origin" inte finns i data (ibland heter det "direction"), justera kolumnnamn här
                available_cols = [col for col in columns_to_pick if col in df.columns]
                return df[available_cols].copy()
            else:
                return pd.DataFrame()
        else:
            # Returnera bara det råa JSON-svaret
            return data

    except requests.exceptions.RequestException as err:
        print(f"Nätverks- eller HTTP-fel: {err}")
        return None


In [85]:
# Stop ID för Göteborg Centralstation
stop_id_gbg_c = 740000002  

# 1) Hämta alla ankomster i en DataFrame
df_arrivals = get_arrivals(stop_id=stop_id_gbg_c, as_dataframe=True)

# 2) Titta på de första raderna
print(df_arrivals.head())

# 3) Räkna hur många ankomster som finns
antal_ankomster = len(df_arrivals)
print(f"Antal ankomster till Göteborg Centralstation: {antal_ankomster}")


                    name                          origin      time        date
0   Länstrafik - Buss X4             Kungälv resecentrum  14:53:00  2025-01-09
1   Länstrafik - Buss X4  Mölnlycke station (Härryda kn)  14:53:00  2025-01-09
2   Länstrafik - Buss 16             Göteborg Lindholmen  14:54:00  2025-01-09
3  Länstrafik - Buss SNU             Stenungsund station  14:54:00  2025-01-09
4   Länstrafik - Buss X1                Partille centrum  14:54:00  2025-01-09
Antal ankomster till Göteborg Centralstation: 218


In [86]:
df_departures = get_timetable(stop_id=740000002, as_dataframe=True)
print(df_departures.head())



                   name                     stop  \
0  Länstrafik - Buss X4        Göteborg Nordstan   
1  Länstrafik - Buss X4        Göteborg Nordstan   
2  Länstrafik - Buss 16        Göteborg Nordstan   
3  Länstrafik - Buss X1        Göteborg Nordstan   
4  Länstrafik - Buss X3  Göteborg Polhemsplatsen   

                      direction        date      time  
0      Höga hallar (Härryda kn)  2025-01-09  14:53:00  
1           Kungälv resecentrum  2025-01-09  14:53:00  
2      Fyrktorget (Göteborg kn)  2025-01-09  14:54:00  
3   Hornkamsgatan (Göteborg kn)  2025-01-09  14:54:00  
4  Särö centrum (Kungsbacka kn)  2025-01-09  14:54:00  


In [87]:
antal_avgångar = len(df_departures)
print(f"Antal avgångar från Göteborg Centralstation: {antal_avgångar}")


Antal avgångar från Göteborg Centralstation: 225


In [88]:
print(df_departures.head())


                   name                     stop  \
0  Länstrafik - Buss X4        Göteborg Nordstan   
1  Länstrafik - Buss X4        Göteborg Nordstan   
2  Länstrafik - Buss 16        Göteborg Nordstan   
3  Länstrafik - Buss X1        Göteborg Nordstan   
4  Länstrafik - Buss X3  Göteborg Polhemsplatsen   

                      direction        date      time  
0      Höga hallar (Härryda kn)  2025-01-09  14:53:00  
1           Kungälv resecentrum  2025-01-09  14:53:00  
2      Fyrktorget (Göteborg kn)  2025-01-09  14:54:00  
3   Hornkamsgatan (Göteborg kn)  2025-01-09  14:54:00  
4  Särö centrum (Kungsbacka kn)  2025-01-09  14:54:00  


In [89]:
# Välj endast de rader där 'name' innehåller texten "Spårväg"
df_trams = df_departures[df_departures["name"].str.contains("Spårväg", case=False, na=False)]


In [90]:
df_trams_filtered = df_trams[["name", "direction", "time"]].copy()


In [91]:
print(df_trams_filtered.head(15))


                       name                           direction      time
15   Länstrafik - Spårväg 6  Kortedala Aprilgatan (Göteborg kn)  14:57:00
30   Länstrafik - Spårväg 6            Göteborg Varmfrontsgatan  15:02:00
42   Länstrafik - Spårväg 6  Kortedala Aprilgatan (Göteborg kn)  15:05:00
66   Länstrafik - Spårväg 6            Göteborg Varmfrontsgatan  15:11:00
78   Länstrafik - Spårväg 6  Kortedala Aprilgatan (Göteborg kn)  15:14:00
109  Länstrafik - Spårväg 6  Kortedala Aprilgatan (Göteborg kn)  15:23:00
110  Länstrafik - Spårväg 6            Göteborg Varmfrontsgatan  15:23:00
147  Länstrafik - Spårväg 6            Göteborg Varmfrontsgatan  15:33:00
155  Länstrafik - Spårväg 6  Kortedala Aprilgatan (Göteborg kn)  15:35:00
177  Länstrafik - Spårväg 6            Göteborg Varmfrontsgatan  15:42:00
187  Länstrafik - Spårväg 6  Kortedala Aprilgatan (Göteborg kn)  15:44:00
218  Länstrafik - Spårväg 6  Kortedala Aprilgatan (Göteborg kn)  15:52:00
224  Länstrafik - Spårväg 6           

In [95]:


result = get_trips(origin_id=740000002, destination_id=740000120)


In [96]:
def get_stops_between(result):
    """
    Tar in JSON-svaret från get_trips()
    och skriver ut namnet på varje hållplats för varje resa (Trip),
    inklusive mellanstopp.
    """
    trips = result.get("Trip", [])
    
    for i, trip in enumerate(trips, start=1):
        print(f"\n=== Trip nr {i} ===")
        
        # Varje trip har "LegList" (en eller flera Leg)
        leg_list = trip.get("LegList", {}).get("Leg", [])
        
        for leg_index, leg in enumerate(leg_list, start=1):
            # Kolla 'Stops' för denna leg
            stops_data = leg.get("Stops", {}).get("Stop", [])
            
            print(f" Leg {leg_index}:")
            for stop in stops_data:
                stop_name = stop.get("name")
                # T.ex. 'Katrineholm Centralstation', 'Malmö Centralstation' etc.
                print("  -", stop_name)


In [97]:
get_stops_between(result)



=== Trip nr 1 ===
 Leg 1:
  - Göteborg Centralstation
  - Mölndal station
  - Kungsbacka station
  - Åsa station (Kungsbacka kn)
  - Varberg station
  - Falkenberg station
  - Halmstad Centralstation
 Leg 2:
  - Halmstad Centralstation
  - Laholm station
  - Båstad station
  - Förslöv station (Båstad kn)
  - Barkåkra station (Ängelholm kn)
  - Ängelholm station
  - Kattarp station (Helsingborg kn)
  - Ödåkra station (Helsingborg kn)
  - Maria station (Helsingborg kn)
  - Helsingborg Centralstation
 Leg 3:
  - Helsingborg Centralstation
  - Ramlösa station (Helsingborg kn)
  - Rydebäck station (Helsingborg kn)
  - Glumslöv station (Landskrona kn)
  - Landskrona station
  - Häljarp station (Landskrona kn)
  - Dösjebro station (Kävlinge kn)
  - Kävlinge station
  - Gunnesbo station (Lund kn)
  - Lund Centralstation

=== Trip nr 2 ===
 Leg 1:
  - Göteborg Centralstation
  - Mölndal station
  - Kungsbacka station
  - Varberg station
  - Falkenberg station
  - Halmstad Centralstation
  - La

In [98]:
def print_trip_stops_and_times(result):
    """
    Tar ett JSON-resultat från get_trips() och skriver ut
    stoppens namn, ankomsttid och avgångstid (om de finns).
    """
    trips = result.get("Trip", [])
    
    for i, trip in enumerate(trips, start=1):
        print(f"\n=== Resa nr {i} ===")
        
        # Kolla alla Leg i denna Trip
        leg_list = trip.get("LegList", {}).get("Leg", [])
        
        for leg_index, leg in enumerate(leg_list, start=1):
            print(f"  Leg {leg_index}: {leg.get('name')}")  # t.ex. "SJ Regional"
            
            # Hämta listan av stopp för denna Leg
            stops_data = leg.get("Stops", {}).get("Stop", [])
            
            # Loop igenom varje stopp och skriv ut tider
            for stop in stops_data:
                stop_name = stop.get("name")
                arr_time = stop.get("arrTime")  # t.ex. "15:32:00"
                dep_time = stop.get("depTime")  # t.ex. "15:33:00"
                
                # Ibland kan arr_time/dep_time vara None om det är start eller slut
                # eller om API:et inte har info. Då skippar vi dem.
                print(f"    - {stop_name}")
                if arr_time:
                    print(f"      Ankomst: {arr_time}")
                if dep_time:
                    print(f"      Avgång:  {dep_time}")


In [99]:
origin_id = 740000001   # Stockholm Centralstation
destination_id = 740000003  # Malmö Centralstation


In [100]:
result = get_trips(origin_id=origin_id, destination_id=destination_id)


In [101]:
print_trip_stops_and_times(result)



=== Resa nr 1 ===
  Leg 1: Snabbtåg 539
    - Stockholm Centralstation
      Avgång:  15:24:00
    - Norrköping Centralstation
      Ankomst: 16:39:00
      Avgång:  16:40:00
    - Linköping Centralstation
      Ankomst: 17:05:00
      Avgång:  17:07:00
    - Nässjö Centralstation
      Ankomst: 17:54:00
      Avgång:  17:55:00
    - Alvesta station
      Ankomst: 18:27:00
      Avgång:  18:28:00
    - Älmhult station
      Ankomst: 18:49:00
      Avgång:  18:50:00
    - Hässleholm Centralstation
      Ankomst: 19:11:00
      Avgång:  19:12:00
    - Lund Centralstation
      Ankomst: 19:40:00
    - Malmö Centralstation
      Ankomst: 19:52:00

=== Resa nr 2 ===
  Leg 1: Snabbtåg 541
    - Stockholm Centralstation
      Avgång:  16:17:00
    - Södertälje Syd station
      Avgång:  16:36:00
    - Norrköping Centralstation
      Ankomst: 17:37:00
      Avgång:  17:38:00
    - Linköping Centralstation
      Ankomst: 18:03:00
      Avgång:  18:05:00
    - Mjölby station
      Ankomst: 18:2

In [102]:
import pandas as pd
import plotly.express as px

def plot_train_stops_on_map(origin_id, destination_id):
    """
    1) Anropar get_trips(origin_id, destination_id) för att hämta resa mellan två orter.
    2) Letar upp alla stopp i första resan (Trip[0]) -> LegList -> Leg -> Stops -> Stop.
    3) Plottar dem i en interaktiv Plotly Express-karta med lat/lon.
    """

    # 1. Hämta JSON-data från din redan definierade get_trips-funktion
    result = get_trips(origin_id=origin_id, destination_id=destination_id)

    # 2. Kolla om vi fick några resor (Trip)
    trips = result.get("Trip", [])
    if not trips:
        print("Inga resor hittades i get_trips-svaret.")
        return
    
    # 3. Välj första resan (eller loopa över alla om du vill)
    trip = trips[0]
    
    # 4. Hämta alla Leg i resan
    leg_list = trip.get("LegList", {}).get("Leg", [])
    
    # 5. Samla info om varje stopp: lat, lon, namn, arrTime, depTime
    stops_data = []
    for leg in leg_list:
    
        
        stops_list = leg.get("Stops", {}).get("Stop", [])
        for stop in stops_list:
            lat = stop.get("lat")
            lon = stop.get("lon")
            name = stop.get("name")
            arr_time = stop.get("arrTime")
            dep_time = stop.get("depTime")

            # Lägg bara till rad om lat/lon finns
            if lat is not None and lon is not None:
                stops_data.append({
                    "StopName": name,
                    "lat": lat,
                    "lon": lon,
                    "arrTime": arr_time,
                    "depTime": dep_time
                })

    # Om listan är tom fick vi inga stopp med koordinater
    if not stops_data:
        print("Hittade inga stopp (eller saknar lat/lon) i resan.")
        return

    # 6. Bygg en DataFrame av alla stopp
    df_stops = pd.DataFrame(stops_data)

    # 7. Plotta interaktiv karta via Plotly Express
    fig = px.scatter_mapbox(
        df_stops,
        lat="lat",
        lon="lon",
        hover_name="StopName",
        hover_data=["arrTime", "depTime"],  # visas när man hovrar
        zoom=5,                             # justera zoom efter behov
        mapbox_style="open-street-map"      # kräver ingen token
    )
    # Layout-inställningar (frivilligt)
    fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
    
    # 8. Visa kartan
    fig.show()


In [104]:
# Exempel: Plotta stopp mellan Göteborg Central och Stockholm Central
# (justera origin/destination efter dina behov)
plot_train_stops_on_map(origin_id=740000002, destination_id=740000003)


In [105]:
import pandas as pd
from datetime import datetime

def show_departure_board_for_stop(stop_id=740015578):
    """
    Hämtar avgångar för en viss hållplats (stop_id),
    filtrerar spårvagnar och bussar, och skriver ut hur många minuter som återstår
    tills de avgår.
    
    stop_id=740015578 -> Göteborg Korsvägen (exempel)
    """
    # 1) Hämta en DataFrame med alla avgångar från hållplatsen
    df_departures = get_timetable(stop_id=stop_id, as_dataframe=True)
    if df_departures.empty:
        print("Inga avgångar hittades för denna hållplats.")
        return

    # 2) Filtrera för att bara få spårvagnar och bussar
    #    I "name" brukar det stå "Länstrafik - Spårväg X" eller "Länstrafik - Buss Y"
    df_filtered = df_departures[
        df_departures["name"].str.contains("Spårväg|Buss", case=False, na=False)
    ].copy()

    # 3) Skapa en kolumn "departure_datetime" av date + time, så vi kan räkna ut minutskillnad
    df_filtered["departure_datetime"] = pd.to_datetime(
        df_filtered["date"] + " " + df_filtered["time"],
        format="%Y-%m-%d %H:%M:%S"
    )

    # 4) Nuvarande tid (lokal tid) - du kan även använda datetime.utcnow() om du vill
    now = datetime.now()

    # 5) Beräkna väntetid i minuter
    #    .total_seconds() // 60 ger heltalsminuter. Använder max(0, ...) för att inte få negativa värden.
    df_filtered["wait_minutes"] = (
        (df_filtered["departure_datetime"] - now).dt.total_seconds() // 60
    ).astype(int).clip(lower=0)  # clip() gör att negativa blir 0 om avgångstiden passerats

    # 6) Skriv ut resultatet i önskat format
    #    Ex: "Spårvagn 8 mot Angered, 2 minuter"
    for _, row in df_filtered.iterrows():
        line_name = row["name"]          # t.ex. "Länstrafik - Spårväg 8"
        direction = row["direction"]     # t.ex. "Angered centrum (Göteborg kn)"
        wait_mins = row["wait_minutes"]  # t.ex. 2
        if wait_mins == 0:
            # Räknas som "avgår nu" om tiden passerats
            wait_text = "avgår nu"
        else:
            wait_text = f"{wait_mins} minuter"

        # (Valfritt) extrahera "Spårväg" och "numret" ur line_name 
        # om du vill enbart skriva "Spårvagn 8" istället för "Länstrafik - Spårväg 8"
        # Här förenklar vi och skriver line_name rakt av
        print(f"{line_name} mot {direction}: {wait_text}")

# -----------------------------------------------------
# Exempel: Anropa funktionen för Göteborg Korsvägen
show_departure_board_for_stop(740015578)


Länstrafik - Buss X4 mot Höga hallar (Härryda kn): avgår nu
Länstrafik - Buss 61 mot Masthugget (Göteborg kn): avgår nu
Länstrafik - Spårväg 8 mot Angered centrum (Göteborg kn): avgår nu
Länstrafik - Buss RÖD mot Lilla Varholmen (Göteborg kn): avgår nu
Länstrafik - Buss X4 mot Kungälv resecentrum: 1 minuter
Länstrafik - Spårväg 8 mot Angered centrum (Göteborg kn): 1 minuter
Länstrafik - Spårväg 5 mot Göteborg Östra sjukhuset: 1 minuter
Länstrafik - Spårväg 4 mot Angered centrum (Göteborg kn): 2 minuter
Länstrafik - Spårväg 8 mot Frölunda torg (Göteborg kn): 2 minuter
Länstrafik - Buss 63 mot Göteborg Heden: 2 minuter
Länstrafik - Buss 18 mot Körkarlens gata (Göteborg kn): 2 minuter
Länstrafik - Buss RÖD mot Önneröd (Härryda kn): 2 minuter
Länstrafik - Buss 100 mot Borås Centralstation: 2 minuter
Länstrafik - Spårväg 4 mot Mölndals Innerstad: 3 minuter
Länstrafik - Buss 63 mot Göteborg Linnéplatsen: 3 minuter
Länstrafik - Buss 18 mot Kallebäck (Göteborg kn): 3 minuter
Länstrafik - Spårv